# Severstal Steel Defect Inspector: Dual-Stage Pipeline

This notebook demonstrates the end-to-end industrial defect inspection framework combining:
1. **Stage 1: The Data Engine (HITL Simulation)**: A pure-normal quantized FAISS memory bank (`IndexIVFPQ`) to mine defect candidates from unlabeled streams without supervision.
2. **Stage 2: The Segmentation Head (Kaggle Scorer)**: A frozen DINOv2 (ViT-B/14) backbone combined with a trainable progressive U-Net decoder to generate pixel-level defect masks and Kaggle `submission.csv`.

In [ ]:
import os
import sys
# Ensure src is discoverable from notebooks directory
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

from src.rle_utils import rle_to_mask, mask_to_rle
from src.dataset import SeverstalDataset, SeverstalUNetDataset
from src.model import DinoUNetDecoder
from src.mine_anomalies import run_data_engine
from src.evaluate import DefectInspector, CLASS_COLORS, CLASS_NAMES

## 1. Dataset Loading and Ground Truth Inspection

In [ ]:
dataset = SeverstalUNetDataset(
    img_dir='../data/severstal/train_images',
    csv_path='../data/severstal/train.csv'
)
print(f"Total annotated images in training set: {len(dataset)}")

# Find a defective image for inspection
defect_idx = next(i for i in range(len(dataset)) if dataset[i]['has_defect'])
sample = dataset[defect_idx]
print(f"Inspecting Defective Image ID: {sample['image_id']}")
print(f"Image Tensor Shape: {sample['image'].shape}")
print(f"4-Channel Target Mask Shape: {sample['mask'].shape}")
print(f"Defect Classes Present: {[c+1 for c in range(4) if torch.sum(sample['mask'][c]) > 0]}")

## 2. Stage 1: The Data Engine (Zero-Shot FAISS Anomaly Mining)

Extracts reference patch tokens from pure normal steel surfaces into a FAISS IVFPQ memory bank. Unlabeled images with nearest-neighbor distance exceeding the threshold are flagged for human review.

In [ ]:
# Run Data Engine simulation
flagged_df = run_data_engine(
    img_dir='../data/severstal/train_images',
    csv_path='../data/severstal/train.csv',
    num_normal=50,
    max_unlabeled=50,
    threshold=100.0,
    output_csv='../data/flagged_for_human_review.csv'
)
print("Top Flagged Candidates for Human Review:")
print(flagged_df.head())

## 3. Stage 2: Supervised DinoUNetDecoder Inference

Loads the trained progressive U-Net decoder weights (`results/best_unet_decoder.pth`) and runs forward inference on test steel images.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DinoUNetDecoder(num_classes=4, device=device)

checkpoint_path = '../results/best_unet_decoder.pth'
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.decoder.load_state_dict(ckpt['decoder_state_dict'])
    print(f"Loaded trained decoder from {checkpoint_path} (Best Val Dice: {ckpt.get('best_val_dice', 0):.4f})")
model.eval()

# Run inference on sample defective image
img_tensor = sample['image'].unsqueeze(0).to(device)
with torch.no_grad():
    preds = model.predict(img_tensor, threshold=0.5)
    pred_masks = preds['binary_masks'].squeeze(0).cpu().numpy()

print("Predicted Defect Mask Shape:", pred_masks.shape)
print("Non-zero mask pixels per class:", [np.sum(pred_masks[c]) for c in range(4)])

## 4. Visualizing Multi-Class Defect Predictions

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), dpi=150)

# Original input image
orig_img = Image.open(os.path.join('../data/severstal/train_images', sample['image_id']))
axes[0].imshow(orig_img)
axes[0].set_title(f"Input Steel Image: {sample['image_id']}", fontsize=11, fontweight='bold')
axes[0].axis('off')

# 4 Class defect channels
for c in range(4):
    axes[c+1].imshow(pred_masks[c], cmap='inferno', vmin=0, vmax=1)
    axes[c+1].set_title(f"Class {c+1} ({CLASS_NAMES.get(c+1, 'Defect')}) Prediction Mask", fontsize=10)
    axes[c+1].axis('off')

plt.tight_layout()
plt.show()

## 5. Kaggle Submission Verification

In [ ]:
if os.path.exists('../submission.csv'):
    sub_df = pd.read_csv('../submission.csv')
    print(f"Total submission rows: {len(sub_df)}")
    print(f"Defect predictions count: {(sub_df['EncodedPixels'].fillna('') != '').sum()}")
    print("\nFirst 8 rows:")
    print(sub_df.head(8))
else:
    print("Run 'python -m src.generate_submission' to generate submission.csv")